# Validation: does the model return sensible comps?

This notebook is evidence of judgement, not a demo. Work through it in order;
each section either produces a number for the README or a reason to go back
and fix something.

Run the pipeline first:

```bash
python run_pipeline.py --stage all
```

The sections are ordered so that the cheap checks that invalidate everything
downstream come first. If feature resolution is broken, nothing below it means
anything.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src import config, paths, similarity

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

features = pd.read_parquet(paths.FEATURES)
percentiles = pd.read_parquet(paths.PERCENTILES)
comps = pd.read_parquet(paths.SIMILARITY)
values = pd.read_parquet(paths.VALUE_PREDICTIONS)

print(f"{len(features):,} player-seasons")
print(features.groupby("position_group").size().to_string())

## 1. Did every feature resolve?

FBref changes table headers from time to time. A feature that silently fails
to resolve is dropped from every group that asked for it, which quietly
degrades the profiles without raising anything.

A non-empty `MISSING` list here is a bug to fix in `src/config.py`, not a
result to work around.

In [ ]:
resolution = pd.read_csv(paths.FEATURE_RESOLUTION)
missing = resolution[resolution["status"] == "MISSING"]

print(f"{len(resolution) - len(missing)}/{len(resolution)} features resolved")
if len(missing):
    print("\nFix these patterns in src/config.py against reports/fbref_columns.txt:")
    display(missing[["feature", "patterns_tried"]])
else:
    print("All features resolved.")

## 2. What did the name join actually achieve?

Report this number in the README. An honest 94% says more about your work
than a silent 100% ever could - and a low rate here means the players you
want to write about may simply be absent.

In [ ]:
summary = json.loads(paths.MATCH_SUMMARY.read_text())
print(json.dumps(summary, indent=2))

review = pd.read_csv(paths.MATCH_REVIEW)
print(f"\n{len(review)} rows need a human decision. Highest-scoring first:")
display(review.head(20)[["fbref_player", "tm_name", "match_score", "fbref_born", "tm_birth_year"]])

In [ ]:
# Did the players you plan to write about survive the join?
# Replace these with the names in your case study.
WATCHLIST = [
    "Alexis Mac Allister",
    "Bruno Guimaraes",
    "Milos Kerkez",
    "Nico Williams",
    "Joshua Kimmich",
]

for name in WATCHLIST:
    rows = features[features["player"].str.contains(name, case=False, na=False)]
    if rows.empty:
        print(f"MISSING  {name}")
    else:
        row = rows.iloc[0]
        value = row["market_value_eur"]
        flag = "" if pd.notna(value) else "   <-- no market value joined"
        print(f"ok       {name:28s} {row['position_group']:3s} {row['team']}{flag}")

## 3. Name your style dimensions

This is the section that turns a model into a story. Read the top and bottom
loadings for the first three components of each group and give each one a
football name.

If PC1 for wingers reads as "shoots and carries a lot" and PC2 as "crosses
rather than cuts inside", you have dimensions a scout recognises - put those
names in the README. If a component is uninterpretable, say so; that is a
finding too.

In [ ]:
loadings = pd.read_csv(paths.PCA_LOADINGS)

def describe(group, n=5):
    chunk = loadings[loadings["position_group"] == group]
    for component in chunk["component"].unique()[:3]:
        block = chunk[chunk["component"] == component].sort_values("loading", ascending=False)
        variance = block["explained_variance_ratio"].iloc[0]
        print(f"\n{group} {component}  ({variance:.1%} of variance)")
        print("  high:", ", ".join(f"{r.feature} {r.loading:+.2f}" for r in block.head(n).itertuples()))
        print("  low: ", ", ".join(f"{r.feature} {r.loading:+.2f}" for r in block.tail(n).itertuples()))

for group in sorted(loadings["position_group"].unique()):
    describe(group)

## 4. The eye test

The real check. Pick players whose game you know well, spread across groups,
and read their top five.

What failure looks like: a target man returning a set of false nines means the
feature set is measuring *output* rather than *role*. A deep-lying playmaker
returning every high-volume passer means the features lack the progression and
zone detail that separates a regista from a metronome.

Write your verdict in the cell below each one. Those verdicts, not the tables,
are what belong in the README.

In [ ]:
def comps_for(name_fragment, n=5, season=None):
    pool = features[features["player"].str.contains(name_fragment, case=False, na=False)]
    if season is not None:
        pool = pool[pool["season"].astype(str) == str(season)]
    if pool.empty:
        print(f"Not found: {name_fragment}")
        return None

    row = pool.nlargest(1, "minutes").iloc[0]
    table = similarity.top_comps(row["player_season_id"], comps, n=n)
    out = table.merge(
        features[["player_season_id", "player", "team", "league", "age",
                  "position_group", "market_value_eur"]],
        left_on="comp_player_season_id", right_on="player_season_id",
        suffixes=("", "_comp"),
    )
    print(f"\n=== {row['player']} ({row['team']}, {row['season']}, {row['position_group']}) ===")
    return out[["rank", "player", "team", "league", "age", "similarity", "market_value_eur"]]

In [ ]:
# Replace with players you can judge confidently.
EYE_TEST = [
    "Rodri",             # DM - should return controllers, not destroyers
    "Virgil van Dijk",   # CB - should return ball-playing CBs
    "Mohamed Salah",     # W  - should return high-volume shooters
    "Erling Haaland",    # ST - should NOT return false nines
    "Trent Alexander",   # FB - should return creative, inverted FBs
]

for name in EYE_TEST:
    table = comps_for(name)
    if table is not None:
        display(table)

**Your verdict on each (write it here):**

- Rodri -
- Virgil van Dijk -
- Mohamed Salah -
- Erling Haaland -
- Trent Alexander-Arnold -

At least one of these should be a miss. Say which and why - an honest failure
case reads as competence, and a README where every comp is perfect reads as
unexamined.

## 5. Self-consistency

The one similarity metric here that is not a matter of taste. The same player
in an adjacent season should rank very high in his own neighbour list. A low
median rank is quantitative evidence the embedding captured the player rather
than the noise.

Put this number in the README.

In [ ]:
validation = json.loads(paths.VALIDATION.read_text())
print(json.dumps(validation, indent=2))

same = comps[comps["same_player"]]
best = same.groupby("player_season_id")["rank"].min()

print(f"\nMedian rank of a player's own other season: {best.median():.0f}")
print(f"In top 5:  {(best <= 5).mean():.1%}")
print(f"In top 10: {(best <= 10).mean():.1%}")

best.plot(kind="hist", bins=25, title="Rank of a player's own other season", figsize=(7, 3));

## 6. Minutes sensitivity

`src/features.py` includes everyone above `INCLUSION_MIN_MINUTES` (600 by
default) and the app defaults its slider to 900. How much does the comp list
change between the two? High churn means the model is reading noise in
low-sample players.

In [ ]:
TIGHT = config.MIN_MINUTES   # 900
LOOSE = config.MIN_MINUTES_SENSITIVITY  # 600

tight_ids = set(features.loc[features["minutes"] >= TIGHT, "player_season_id"])

rows = []
for player_season_id in list(tight_ids)[:400]:
    loose_top5 = set(
        similarity.top_comps(player_season_id, comps, n=5)["comp_player_season_id"]
    )
    tight_only = comps[
        (comps["player_season_id"] == player_season_id)
        & (~comps["same_player"])
        & (comps["comp_player_season_id"].isin(tight_ids))
    ].nsmallest(5, "rank")
    overlap = len(loose_top5 & set(tight_only["comp_player_season_id"]))
    rows.append(overlap)

print(f"Mean overlap of top-5 comps, {LOOSE}+ pool vs {TIGHT}+ pool: {np.mean(rows):.2f} / 5")
print("Below about 3/5 suggests the extra low-minutes players are noisy enough to matter.")

## 7. Is the value model learning anything but league and age?

The characteristic failure: the undervalued shortlist is all 33-year-olds in
the cheapest league in your sample, because the model has learnt league and
age and nothing else.

Check the composition of the shortlist against the composition of the pool. If
they diverge sharply on league or age, the residuals are mostly a league
effect and you should read them inside a filter.

In [ ]:
metrics = json.loads(paths.VALUE_METRICS.read_text())
print(json.dumps(metrics, indent=2))

In [ ]:
flagged = values[values["undervalued"]]

comparison = pd.DataFrame({
    "pool": values["league"].value_counts(normalize=True),
    "flagged": flagged["league"].value_counts(normalize=True),
}).fillna(0).round(3)
print("League composition:")
display(comparison)

print("\nAge:")
display(pd.DataFrame({
    "pool": values["age"].describe(),
    "flagged": flagged["age"].describe(),
}).round(1))

print("\nTop 15 by residual:")
display(
    flagged.nsmallest(15, "log_residual")[
        ["player", "team", "league", "age", "market_value_eur",
         "predicted_value_eur", "value_ratio"]
    ]
)

## 8. Known transfers - the honest failure test

Take replacement signings clubs actually made and check whether the model would
have surfaced the replacement as a comparable to the departing player.

Some will miss. Those misses, explained, are more persuasive than the hits:
they show you know what the model cannot see - injury history, contract
clauses, attitude, whether a profile that works in a low block survives in a
high line.

In [ ]:
# (departed, replacement) pairs - use transfers you can reason about.
TRANSFER_PAIRS = [
    ("Kim Min-jae", "Josko Gvardiol"),
    ("Sadio Mane", "Luis Diaz"),
    ("Christian Eriksen", "Bruno Guimaraes"),
]

for departed, replacement in TRANSFER_PAIRS:
    pool = features[features["player"].str.contains(departed, case=False, na=False)]
    if pool.empty:
        print(f"{departed}: not in sample")
        continue
    row = pool.nlargest(1, "minutes").iloc[0]
    ranked = comps[
        (comps["player_season_id"] == row["player_season_id"]) & (~comps["same_player"])
    ].merge(
        features[["player_season_id", "player"]],
        left_on="comp_player_season_id", right_on="player_season_id",
        suffixes=("", "_comp"),
    )
    hit = ranked[ranked["player"].str.contains(replacement, case=False, na=False)]
    if hit.empty:
        print(f"MISS  {departed} -> {replacement}: not in the top {config.N_NEIGHBOURS}")
    else:
        print(f"HIT   {departed} -> {replacement}: rank {int(hit.iloc[0]['rank'])}")

---

## What goes in the README

Fill these in from the cells above:

| Claim | Your number |
| --- | --- |
| Player-seasons in the sample | |
| Name match rate | |
| Median rank of a player's own other season | |
| Value model median absolute % error (CV, grouped by player) | |
| Smallest position group | |
| Named PCA dimensions, per group | |
| At least one honest failure case | |